<a href="https://colab.research.google.com/github/Murphy2666/deep-learning-notes/blob/main/001_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
tf.__version__

'2.20.0'

# tensorflow quick start for beginners #

https://www.tensorflow.org/tutorials/quickstart/beginner

### input layer ###

In [ ]:
mnist = tf.keras.datasets.mnist
(x_train, y_train),(x_test,y_test)=mnist.load_data()
x_train, x_test = x_train/255.0, x_test/255.0

print("x_train shape is ", x_train.shape, "x_test shape is ", x_test.shape)
print("y_train shape is ", y_train.shape, "y_test shape is ", y_test.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
x_train shape is  (60000, 28, 28) x_test shape is  (10000, 28, 28)
y_train shape is  (60000,) y_test shape is  (10000,)


### dense layer ###

定义向前传播 sequential表示按照顺序运行
* **tf.keras.layers.Flatten**: 一个样本的tensor压成一行，因为后面dense一般处理一维数据（不包括第一个batch，否则dense一般处理二维数据）
  * x_train.shape: (60000,28,28) -> (60000,784)

* **tf.keras.layers.Dense(128, activation="relu)**: 设置128个神经元，对一个样本做128次线性拟合
  * x_train*W^t +b \
  x_train.shape=[60000, 784]\
  W^t.shape=[784, 128]\
  b=[1,128] -> [60000, 128] but broadcasting it across it all, making each row the same\
  relu the output: shape is [60000, 128]
  * 默认用Xavier的uniform distribution:
  $$W \sim U \left[ -\frac{\sqrt{6}}{\sqrt{n_j + n_{j+1}}}, \frac{\sqrt{6}}{\sqrt{n_j + n_{j+1}}} \right]$$
  在这个例子里 j是输入层\
  nj=784\
  nj+1=128\
  $b$是0向量

* **tf.keras.layers.Dropout**: randomly set input units to 0 to prevent overfitting, the rest are scaled up by 1/(1-rate) to maintain the **expected** sum \
here it's 20% randomly set to 0 and the rest scaled up by 1/(1-0.2)=1.25 \
math: E=(**0.2** * 0)+(**0.8** * 1.25 * x)=x\
x is the original E(x)

* **tf.keras.layers.Dense(10)**:
  * 10 neuros, ouput 10 values per sample
  input: X_input is [60000, 128]
  X_input*W^t_output + b_output
  kernal is [128, 10]\
  bias is [1, 10] broadcasting it to [60000, 10]\
  get 10 nums\
  最终得到60000个长度为10的预测得分logits, 未归一化
  （对应y是0-9）

In [ ]:
model=tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28,28))
    ,tf.keras.layers.Dense(128,activation='relu')
    ,tf.keras.layers.Dropout(0.2)
    ,tf.keras.layers.Dense(10)
])
print(model(x_train).shape)
print(model(x_train)[:1])

(60000, 10)
tf.Tensor(
[[ 0.9938541   0.6171017  -0.30382758 -0.07640452 -0.76303315 -0.80025166
  -0.17924611 -0.12962702  0.7545255  -0.03285815]], shape=(1, 10), dtype=float32)


### example for 1st forward propagation ###

predictions用softmax预测10分类中是哪一个

In [ ]:
#第一次向前传播 1st forward propagation
predictions = model(x_train)
print(tf.nn.softmax(predictions).shape)
tf.nn.softmax(predictions).numpy()

(60000, 10)


array([[2.59656164e-12, 2.37204752e-11, 1.31579247e-09, ...,
        6.26148411e-10, 3.65379588e-10, 7.06428427e-09],
       [9.99992669e-01, 9.18466692e-11, 5.06405104e-06, ...,
        2.49485925e-08, 1.63445009e-08, 1.65278789e-06],
       [4.00710505e-08, 3.85609928e-06, 1.25445577e-03, ...,
        4.64181176e-06, 3.06126390e-06, 3.26674839e-04],
       ...,
       [6.06487915e-11, 5.71268310e-11, 1.38087363e-13, ...,
        6.14411161e-11, 2.75676122e-08, 2.66434284e-07],
       [1.88314971e-06, 5.40033852e-07, 1.49630714e-05, ...,
        3.86734712e-07, 9.00554824e-07, 2.68280473e-08],
       [3.09795956e-04, 1.57329310e-07, 9.08223228e-05, ...,
        1.01761725e-05, 9.99474466e-01, 1.20634086e-05]], dtype=float32)

loss用交叉熵SparseCategoricalCrossentropy\
diff from CategoricalCrossentropy\
CategoricalCrossentropy处理one-hot矩阵\
SparseCategoricalCrossentropy直接处理标签，把标签值做index从predictions里拿数

eg.\
y_true=[1,2] one-hot是[[0,1,0],[0,0,1]]
SparseCategoricalCrossentropy 会直接抓取：
  - 第 0 个样本第 1 列: predictions[0][1]
  - 第 1 个样本第 2 列: predictions[1][2]
损失为: -ln(predictions[0][1]) - ln(predictions[1][2])，其他项因为 0 * ln(...) 被忽略。
最后总和取平均数

因为第一次向前传播，交叉熵loss理论上会很接近随机分类后交叉熵的结果

This untrained model gives probabilities close to random (1/10 for each class), so the initial loss should be close to (1/60000)*(60000*log(1/10))
-tf.math.log(1/10) ~= 2.3

## loss & eta

In [ ]:
# 定义用哪个loss函数
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
# 整体的loss
print("整体的loss是", loss_fn(y_train, predictions).numpy())
# 第一个loss
print("第一个sample的loss是", loss_fn(y_train[:1], predictions[:1]).numpy())

整体的loss是 2.4023416
第一个sample的loss是 2.963357


绑定**学习率eta, loss方法, 用accuracy评估模型好坏**到self(也就是model这个实例对象上)\
后面 model.fit 会自动去 self 上读取 compile 存好的 loss 和 optimizer
* adam的default  $\eta = 0.001$

---
model是上面model=tf.keras.models.Sequential(...)得到的，继承自tf.keras.Model

需要调用这个对象继承来的compile方法



In [ ]:
model.compile(optimizer='adam', loss=loss_fn, metrics=['accuracy'])


## train & fit

默认batch size是32\
等价于一个epoch会更新(全部sample/batch_size)=60000/32=1875次参数\
每过32个sample更新一次参数\
如果遇到不能整除的sample size比如60001, 最后一次更新就1个样本

下面5个epoch 每个进度条显示**1875/1875**

In [ ]:
model.fit(x_train, y_train, epochs=5)


Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9785 - loss: 0.0659
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9818 - loss: 0.0560
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9829 - loss: 0.0524
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9842 - loss: 0.0476
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9859 - loss: 0.0435


## Evaluation ##

In [ ]:
model.evaluate(x_test,  y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9797 - loss: 0.0726


[0.07264929264783859, 0.9797000288963318]

## furthermore: if wanna return a probabilty ##

再用Sequential包一层softmax

In [ ]:
probability_model=tf.keras.models.Sequential([model, tf.keras.layers.Softmax()])
probability_model(x_test[:5])

<tf.Tensor: shape=(5, 10), dtype=float32, numpy=
array([[5.9715560e-10, 1.6010122e-09, 9.8692809e-08, 1.0038607e-04,
        1.1834362e-13, 1.6536308e-07, 2.8728046e-15, 9.9989939e-01,
        2.4474915e-08, 2.8591815e-08],
       [2.8865019e-10, 3.0722364e-07, 9.9999940e-01, 2.5146343e-07,
        3.1142164e-16, 8.7122549e-09, 3.0129026e-09, 2.7627081e-16,
        3.9575486e-08, 2.0675464e-17],
       [2.4922695e-09, 9.9987578e-01, 1.5488754e-06, 3.8418407e-06,
        4.6698760e-06, 3.3188614e-06, 7.8296409e-07, 1.0021468e-04,
        9.7620468e-06, 2.2004196e-08],
       [9.9999082e-01, 1.2169188e-07, 6.0378688e-06, 6.0056768e-09,
        1.4487193e-08, 4.0407818e-07, 1.9315992e-06, 5.1120912e-07,
        1.1887781e-08, 6.1588864e-08],
       [2.2527633e-08, 4.1457518e-11, 3.7953239e-08, 2.7844493e-09,
        9.9897170e-01, 1.6035210e-07, 1.3670685e-08, 1.2821998e-06,
        2.1200538e-07, 1.0267214e-03]], dtype=float32)>